In [2]:
import sys
import os
import subprocess
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import typing as t

# =========================================================
# 1. LOCAL PATH CONNECTION
# =========================================================

project_root = '/Users/andaks/Columbia/Columbia Fall25/RL/Waymo-Agent'
paths_to_add = [project_root]

print("Configuring local paths...")
for p in paths_to_add:
    if os.path.exists(p):
        if p not in sys.path:
            sys.path.append(p)
        print(f"  [OK] Added to sys.path: {p}")
    else:
        print(f"  [ERROR] Path not found: {p}")

if os.path.exists(project_root):
    os.chdir(project_root)

# =========================================================
# 2. DEPENDENCY CHECK
# =========================================================
required_libs = ['osmnx', 'gymnasium', 'geopandas', 'networkx']

print("\nChecking dependencies...")
for lib in required_libs:
    try:
        __import__(lib)
    except ImportError:
        print(f"  [MISSING] {lib} is NOT installed. Installing...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", lib])
        except:
            sys.exit(1)

# =========================================================
# 3. IMPORTS
# =========================================================

try:
    import osmnx 
    if "waymo_agent" in str(osmnx.__file__):
        print("\n[CRITICAL WARNING] 'import osmnx' loaded your local folder, not the library!")
        print("This will cause errors. Ensure only Project Root is in sys.path.")
    
    from waymo_agent.graph_env.ENV import RideShareEnv
    from waymo_agent.data_classes.dataclasses import EnvConfig 
    print("  [OK] RideShareEnv imported successfully.")

except ImportError as e:
    print(f"\nCRITICAL IMPORT ERROR: {e}")
    sys.exit(1)

# ---------------------------------------------------------
# 4. CONFIGURATION & DEVICE
# ---------------------------------------------------------
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# Helper function to handle the NumPy 2.x / PyTorch incompatibility
def to_numpy(tensor):
    try:
        return tensor.detach().cpu().numpy()
    except RuntimeError:
        return np.array(tensor.detach().cpu().tolist())

# ---------------------------------------------------------
# 5. ADAPTER (Gymnasium -> PPO)
# ---------------------------------------------------------
class RideShareAdapter:
    """
    Adapts the RideShareEnv (Gymnasium) to work with the PPO logic.
    Handles the translation between PPO Discrete actions and Env Continuous/Dict actions.
    """
    def __init__(self, config=None):
        print("  [DEBUG] Initializing RideShareEnv...")
        try:
            self.env = RideShareEnv(config=config)
            print("  [DEBUG] RideShareEnv initialized.")
        except Exception as e:
            print(f"  [ERROR] Error inside RideShareEnv.__init__: {e}")
            raise e
             
        self.num_vehicles = self.env.num_vehicles
        # We need max_pending_requests to construct valid zero-actions for dispatch/prices
        self.max_pending = self.env.config.max_pending_requests
        
        print("  [DEBUG] Performing dummy reset to determine observation shape...")
        obs, _ = self.env.reset()
        
        if obs is None:
            print("  [WARNING] env.reset() returned None for observation! Using fallback empty dict.")
            obs = {}

        self.processed_obs_shape = self._process_observation(obs).shape[1]
        
        # We force Action Dim = 5 (Discrete) for the PPO
        # We will map these 5 discrete actions to continuous vectors in step()
        self.action_dim = 5 

        print(f"Adapter Ready: {self.num_vehicles} Vehicles, Obs Dim: {self.processed_obs_shape}, Act Dim: {self.action_dim}")

    def _process_observation(self, raw_obs: t.Dict[str, np.ndarray]) -> np.ndarray:
        if raw_obs is None:
            raw_obs = {}

        # --- Shared Global Context ---
        globs = raw_obs.get('globals', np.zeros(7))
        sd_ratio = raw_obs.get('supply_demand_ratio', np.zeros(1))
        
        pending_reqs = raw_obs.get('pending_requests')
        if pending_reqs is None: pending_reqs = np.array([])
        else: pending_reqs = pending_reqs.flatten()

        pricing_mask = raw_obs.get('pricing_mask')
        if pricing_mask is None: pricing_mask = np.array([])
        else: pricing_mask = pricing_mask.flatten()
        
        # Combine shared features and broadcast to all vehicles
        shared_features = np.concatenate([globs, sd_ratio, pending_reqs, pricing_mask])
        global_batch = np.tile(shared_features, (self.num_vehicles, 1))

        # --- Per-Agent Local Context ---
        vehicles_state = raw_obs.get('vehicles', np.zeros((self.num_vehicles, 7)))
        active_rides = raw_obs.get('active_rides', np.zeros((self.num_vehicles, 8)))
        
        dispatch_mask = raw_obs.get('dispatch_mask', np.zeros(self.num_vehicles))
        if dispatch_mask.ndim == 1:
            dispatch_mask = dispatch_mask.reshape(-1, 1)

        local_batch = np.concatenate([vehicles_state, active_rides, dispatch_mask], axis=1)

        final_obs = np.concatenate([global_batch, local_batch], axis=1)
        return final_obs.astype(np.float32)

    def reset(self):
        obs, _ = self.env.reset()
        if obs is None: obs = {} 
        return self._process_observation(obs)

    def step(self, action_list):
        """
        Convert PPO's list of discrete actions (0-4) into the specific 
        NumPy arrays required by action_mixin.py.
        """
        # 1. Map Discrete Actions to Continuous Reposition Vectors (dx, dy)
        # 0: Stay, 1: Up, 2: Down, 3: Left, 4: Right
        reposition_vectors = []
        for a in action_list:
            if a == 0: vec = [0.0, 0.0]
            elif a == 1: vec = [0.0, 1.0]
            elif a == 2: vec = [0.0, -1.0]
            elif a == 3: vec = [-1.0, 0.0]
            elif a == 4: vec = [1.0, 0.0]
            else: vec = [0.0, 0.0]
            reposition_vectors.append(vec)
        
        # Reposition must be (Num_Vehicles, 2)
        reposition_arr = np.array(reposition_vectors, dtype=np.float32)

        # 2. Create Dummy Arrays for Prices and Dispatch (Required by ActionSpace)
        # Prices: (max_pending_requests,)
        # Dispatch: (max_pending_requests, num_vehicles)
        prices_arr = np.zeros(self.max_pending, dtype=np.float32)
        dispatch_arr = np.zeros((self.max_pending, self.num_vehicles), dtype=np.int8)

        # 3. Construct the Action Dictionary
        env_action = {
            "prices": prices_arr,
            "reposition": reposition_arr,
            "dispatch": dispatch_arr
        }

        obs, reward, terminated, truncated, info = self.env.step(env_action)
        if obs is None: obs = {} 
        
        rewards = np.full(self.num_vehicles, reward, dtype=np.float32)
        done = terminated or truncated
        
        next_obs_batch = self._process_observation(obs)
        return next_obs_batch, rewards, done, info

# ---------------------------------------------------------
# 6. PPO MODEL CLASSES
# ---------------------------------------------------------

class Policy(object):
    def __init__(self, obssize, actsize, lr, device):
        self.device = device
        self.actsize = actsize
        self.model = torch.nn.Sequential(
            torch.nn.Linear(obssize, 256),
            torch.nn.ReLU(),
            torch.nn.Linear(256, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, actsize)
        ).to(self.device) 
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)

    def compute_prob(self, states):
        states = torch.FloatTensor(states).to(self.device)
        logits = self.model(states)
        prob = torch.nn.functional.softmax(logits, dim=-1)
        return to_numpy(prob)

    def _to_one_hot(self, y, num_classes):
        scatter_dim = len(y.size())
        y_tensor = y.view(*y.size(), -1)
        zeros = torch.zeros(*y.size(), num_classes, dtype=y.dtype, device=self.device)
        return zeros.scatter(scatter_dim, y_tensor, 1)

    def train(self, states, actions, Qs):
        states = torch.FloatTensor(states).to(self.device)
        actions = torch.LongTensor(actions).to(self.device)
        Qs = torch.FloatTensor(Qs).to(self.device)

        logits = self.model(states)
        prob = torch.nn.functional.softmax(logits, dim=-1)

        action_onehot = self._to_one_hot(actions, self.actsize)
        prob_selected = torch.sum(prob * action_onehot, axis=-1)
        prob_selected += 1e-8

        log_prob_selected = torch.log(prob_selected)
        loss = -torch.mean(Qs * log_prob_selected)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return to_numpy(loss)


class ValueFunction(object):
    def __init__(self, obssize, lr, device):
        self.device = device
        self.model = torch.nn.Sequential(
            torch.nn.Linear(obssize, 256),
            torch.nn.ReLU(),
            torch.nn.Linear(256, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 1)
        ).to(self.device)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)

    def compute_values(self, states):
        states = torch.FloatTensor(states).to(self.device)
        return to_numpy(self.model(states))

    def train(self, states, targets):
        states = torch.FloatTensor(states).to(self.device)
        targets = torch.FloatTensor(targets).to(self.device)
        v_preds = self.model(states)
        loss = torch.nn.functional.mse_loss(v_preds, targets.unsqueeze(1))
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return to_numpy(loss)

def discounted_rewards(r, gamma):
    discounted_r = np.zeros_like(r, dtype=np.float32)
    running_sum = 0
    for i in reversed(range(0, len(r))):
        discounted_r[i] = running_sum * gamma + r[i]
        running_sum = discounted_r[i]
    return list(discounted_r)

# ---------------------------------------------------------
# 7. MAIN EXECUTION LOOP
# ---------------------------------------------------------

# Settings
lr_actor = 3e-4
lr_critic = 1e-3
gamma = 0.99
iterations = 200 
numtrajs = 5 

# Initialize Environment
env_adapter = None
try:
    env_adapter = RideShareAdapter()
except Exception as e:
    print(f"Environment Init Failed: {e}")
    import traceback
    traceback.print_exc()

if env_adapter:
    obssize = env_adapter.processed_obs_shape
    actsize = env_adapter.action_dim
    num_vehicles = env_adapter.num_vehicles

    # Initialize Networks
    actor = Policy(obssize, actsize, lr_actor, device)
    baseline = ValueFunction(obssize, lr_critic, device)

    print("\nStarting PPO Training on RideShareEnv...")
    history_rewards = []

    for ite in range(iterations):
        OBS_BUFFER = []
        ACTS_BUFFER = []
        VAL_BUFFER = []

        total_batch_reward = 0

        # Collect Trajectories
        for num in range(numtrajs):
            car_obss = [[] for _ in range(num_vehicles)]
            car_acts = [[] for _ in range(num_vehicles)]
            car_rews = [[] for _ in range(num_vehicles)]

            obs_batch = env_adapter.reset()
            done = False
            steps = 0
            
            max_steps = 500

            while not done and steps < max_steps:
                # 1. Store state
                for i in range(num_vehicles):
                    car_obss[i].append(obs_batch[i])

                # 2. Action Probabilities
                probs = actor.compute_prob(obs_batch)

                # 3. Sample Actions
                actions = []
                for i in range(num_vehicles):
                    p_vec = probs[i].flatten()
                    if np.isnan(p_vec).any(): p_vec = np.ones_like(p_vec) / len(p_vec)
                    a = np.random.choice(actsize, p=p_vec)
                    actions.append(a)
                    car_acts[i].append(a)

                # 4. Step
                next_obs_batch, rewards, done, _ = env_adapter.step(actions)

                # 5. Store rewards
                for i in range(num_vehicles):
                    car_rews[i].append(rewards[i])

                obs_batch = next_obs_batch
                steps += 1

            # End of Episode Processing
            ep_total = 0
            for i in range(num_vehicles):
                ep_total += np.sum(car_rews[i])
                dis_r = discounted_rewards(car_rews[i], gamma)
                
                OBS_BUFFER.extend(car_obss[i])
                ACTS_BUFFER.extend(car_acts[i])
                VAL_BUFFER.extend(dis_r)

            total_batch_reward += (ep_total / num_vehicles) 

        # Reporting
        avg_ep_reward = total_batch_reward / numtrajs
        history_rewards.append(avg_ep_reward)
        
        if ite % 5 == 0:
            print(f"Iteration {ite}, Avg Fleet Reward: {avg_ep_reward:.4f}")

        # PPO Update
        obs_train = np.array(OBS_BUFFER)
        val_train = np.array(VAL_BUFFER)
        acts_train = np.array(ACTS_BUFFER)
        
        if len(obs_train) == 0: continue

        baseline.train(obs_train, val_train)
        
        v_preds = baseline.compute_values(obs_train).flatten()
        advantages = val_train - v_preds
        if advantages.std() > 1e-8:
            advantages = (advantages - advantages.mean()) / advantages.std()

        actor.train(obs_train, acts_train, advantages)

    # Plot
    plt.figure(figsize=(10, 5))
    plt.plot(history_rewards)
    plt.title("PPO Training on RideShareEnv")
    plt.xlabel("Iteration")
    plt.ylabel("Average Fleet Reward")
    plt.grid(True)
    plt.show()

Configuring local paths...
  [OK] Added to sys.path: /Users/andaks/Columbia/Columbia Fall25/RL/Waymo-Agent

Checking dependencies...
  [OK] RideShareEnv imported successfully.
Using device: cpu
  [DEBUG] Initializing RideShareEnv...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


  [DEBUG] RideShareEnv initialized.
  [DEBUG] Performing dummy reset to determine observation shape...
  [WARNING] env.reset() returned None for observation! Using fallback empty dict.
Adapter Ready: 77 Vehicles, Obs Dim: 24, Act Dim: 5

Starting PPO Training on RideShareEnv...
Iteration 0, Avg Fleet Reward: 0.0000
Iteration 5, Avg Fleet Reward: 0.0000
Iteration 10, Avg Fleet Reward: 0.0000


KeyboardInterrupt: 